In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets pillow kagglehub scikit-learn

In [ ]:
import os
import torch
import pandas as pd

from PIL import Image

from datasets import Dataset

from transformers import (

    AutoProcessor,

    Qwen2_5_VLForConditionalGeneration,

    BitsAndBytesConfig,

    TrainingArguments,

    Trainer
)

from peft import (

    LoraConfig,

    get_peft_model
)

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    confusion_matrix
)

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "nrizwan/mami-dataset"
)

print(dataset_path)

In [ ]:
print(os.listdir(dataset_path))

In [ ]:
for root, dirs, files in os.walk(dataset_path):

    print(root)

    print(files[:5])

    print("=" * 50)

In [ ]:
# ============================================================
# STEP 6 — DEFINE CORRECT PATHS
# ============================================================

DATA_DIR = os.path.join(
    dataset_path,
    "MAMI_Dataset",
    "data"
)

TRAIN_FILE = os.path.join(
    DATA_DIR,
    "train.tsv"
)

DEV_FILE = os.path.join(
    DATA_DIR,
    "validation.tsv"
)

TEST_FILE = os.path.join(
    DATA_DIR,
    "test.tsv"
)

TRAIN_IMAGE_DIR = os.path.join(
    DATA_DIR,
    "MAMI_2022_images",
    "training_images"
)

TEST_IMAGE_DIR = os.path.join(
    DATA_DIR,
    "MAMI_2022_images",
    "test_images"
)

print(TRAIN_FILE)

print(DEV_FILE)

print(TEST_FILE)

print(TRAIN_IMAGE_DIR)

print(TEST_IMAGE_DIR)

In [ ]:
train_df = pd.read_csv(
    TRAIN_FILE,
    sep="\t"
)

dev_df = pd.read_csv(
    DEV_FILE,
    sep="\t"
)

test_df = pd.read_csv(
    TEST_FILE,
    sep="\t"
)

print(train_df.head())

In [ ]:
print(train_df.columns)

In [ ]:
label_column = "label"

train_df[label_column] = train_df[
    label_column
].astype(int)

dev_df[label_column] = dev_df[
    label_column
].astype(int)

test_df[label_column] = test_df[
    label_column
].astype(int)

In [ ]:
train_df = train_df[:200]

dev_df = dev_df[:100]

test_df = test_df[:100]

print(len(train_df))

print(len(dev_df))

print(len(test_df))

In [ ]:
def format_sample(row, image_dir):

    image_path = os.path.join(

        image_dir,

        str(row["file_name"])
    )

    prompt = f"""
Analyze this meme.

Text:
{row['text']}

Classify it as:
- misogynous
- non-misogynous

Answer only one word.
"""

    return {

        "image": image_path,

        "prompt": prompt,

        "label": int(
            row["label"]
        )
    }


train_formatted = [

    format_sample(
        x,
        TRAIN_IMAGE_DIR
    )

    for _, x in train_df.iterrows()
]

dev_formatted = [

    format_sample(
        x,
        TRAIN_IMAGE_DIR
    )

    for _, x in dev_df.iterrows()
]

test_formatted = [

    format_sample(
        x,
        TEST_IMAGE_DIR
    )

    for _, x in test_df.iterrows()
]

In [ ]:
print(train_formatted[0])

print(
    os.path.exists(
        train_formatted[0]["image"]
    )
)

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(
    train_formatted
)

dev_dataset = Dataset.from_list(
    dev_formatted
)

test_dataset = Dataset.from_list(
    test_formatted
)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_use_double_quant=True
)

processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(

    MODEL_NAME,

    quantization_config=bnb_config,

    device_map="auto"
)

In [ ]:
lora_config = LoraConfig(

    r=8,

    lora_alpha=16,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

In [ ]:
def preprocess(example):

    image = Image.open(
        example["image"]
    ).convert("RGB")

    messages = [
        {
            "role": "user",

            "content": [

                {
                    "type": "image",
                    "image": image
                },

                {
                    "type": "text",
                    "text": example["prompt"]
                }
            ]
        }
    ]

    text = processor.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True
    )

    inputs = processor(

        text=[text],

        images=[image],

        padding=True,

        return_tensors="pt"
    )

    inputs = {

        k: v.squeeze(0)

        for k, v in inputs.items()
    }

    answer_text = (

        "misogynous"

        if example["label"] == 1

        else "non-misogynous"
    )

    answer_tokens = processor.tokenizer(

        answer_text,

        add_special_tokens=False
    ).input_ids

    input_ids = inputs[
        "input_ids"
    ].tolist()

    attention_mask = inputs[
        "attention_mask"
    ].tolist()

    input_ids = input_ids + answer_tokens

    attention_mask = (
        attention_mask +
        [1] * len(answer_tokens)
    )

    labels = (
        [-100] * len(inputs["input_ids"])
        + answer_tokens
    )

    return {

        "input_ids": torch.tensor(input_ids),

        "attention_mask": torch.tensor(attention_mask),

        "labels": torch.tensor(labels),

        "pixel_values": inputs["pixel_values"],

        "image_grid_thw": inputs[
            "image_grid_thw"
        ]
    }

In [ ]:
train_dataset = train_dataset.map(
    preprocess
)

dev_dataset = dev_dataset.map(
    preprocess
)

In [ ]:
def collate_fn(batch):

    input_ids = []

    attention_mask = []

    labels = []

    pixel_values = []

    image_grid_thw = []

    for x in batch:

        ids = x["input_ids"]

        mask = x["attention_mask"]

        lbl = x["labels"]

        pv = x["pixel_values"]

        grid = x["image_grid_thw"]

        if isinstance(ids, list):
            ids = torch.tensor(ids)

        if isinstance(mask, list):
            mask = torch.tensor(mask)

        if isinstance(lbl, list):
            lbl = torch.tensor(lbl)

        if isinstance(pv, list):
            pv = torch.tensor(pv)

        if isinstance(grid, list):
            grid = torch.tensor(grid)

        input_ids.append(ids)

        attention_mask.append(mask)

        labels.append(lbl)

        pixel_values.append(pv)

        image_grid_thw.append(grid)

    input_ids = torch.nn.utils.rnn.pad_sequence(

        input_ids,

        batch_first=True,

        padding_value=processor.tokenizer.pad_token_id
    )

    attention_mask = torch.nn.utils.rnn.pad_sequence(

        attention_mask,

        batch_first=True,

        padding_value=0
    )

    labels = torch.nn.utils.rnn.pad_sequence(

        labels,

        batch_first=True,

        padding_value=-100
    )

    pixel_values = torch.cat(
        pixel_values,
        dim=0
    )

    # ✅ FIX HERE
    image_grid_thw = torch.stack(
        image_grid_thw
    )

    return {

        "input_ids": input_ids,

        "attention_mask": attention_mask,

        "labels": labels,

        "pixel_values": pixel_values,

        "image_grid_thw": image_grid_thw
    }

In [ ]:
training_args = TrainingArguments(

    output_dir="./qwen_mami_output",

    per_device_train_batch_size=1,

    per_device_eval_batch_size=1,

    gradient_accumulation_steps=1,

    num_train_epochs=15,

    learning_rate=2e-4,

    fp16=True,

    logging_steps=5,

    logging_strategy="steps",

    disable_tqdm=False,

    save_strategy="no",

    report_to="none"
)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=dev_dataset,

    data_collator=collate_fn
)

In [ ]:
trainer.train()

In [ ]:
def predict(sample):

    image = Image.open(
        sample["image"]
    ).convert("RGB")

    messages = [
        {
            "role": "user",

            "content": [

                {
                    "type": "image",
                    "image": image
                },

                {
                    "type": "text",
                    "text": sample["prompt"]
                }
            ]
        }
    ]

    text = processor.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True
    )

    inputs = processor(

        text=[text],

        images=[image],

        return_tensors="pt"
    )

    inputs = {

        k: v.to(model.device)

        for k, v in inputs.items()
    }

    generated_ids = model.generate(

        **inputs,

        max_new_tokens=10
    )

    output = processor.batch_decode(

        generated_ids,

        skip_special_tokens=True
    )[0]

    output = output.lower()

    if "misogynous" in output:

        return 1

    else:

        return 0

In [ ]:
import time

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    confusion_matrix,

    classification_report
)

y_true = []

y_pred = []

inference_times = []

for sample in test_formatted:

    start_time = time.time()

    prediction = predict(sample)

    end_time = time.time()

    inference_time = (
        end_time - start_time
    )

    inference_times.append(
        inference_time
    )

    y_true.append(
        sample["label"]
    )

    y_pred.append(
        prediction
    )

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred
)

recall = recall_score(
    y_true,
    y_pred
)

f1 = f1_score(
    y_true,
    y_pred
)

cm = confusion_matrix(
    y_true,
    y_pred
)

average_inference_time = sum(
    inference_times
) / len(inference_times)

print("Accuracy:", accuracy)

print("Precision:", precision)

print("Recall:", recall)

print("F1-score:", f1)

print(
    "Average Inference Time (seconds):",
    average_inference_time
)

print("Confusion Matrix:")

print(cm)


report = classification_report(

    y_true,

    y_pred,

    target_names=[

        "Non-Misogynous",

        "Misogynous"
    ]
)

print(report)